In [1]:
import sys
import os

In [2]:
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))

if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
        
from core.models import GetModel
from core.datasets import DatasetCN
import json

In [4]:
def GetModelFromConfig(path):
    config_path =  "config.json"
    with open(config_path, "r") as f:
        config = json.load(f)
                
    model = GetModel(config)
    
    return model, config["experiment"]["N"]

In [5]:
model, N = GetModelFromConfig("")

model_path = r"checkpoint_epoch_7000.pt"

checkpoint = torch.load(model_path, map_location="cpu")

model_weights = checkpoint["model_state_dict"]
model.load_state_dict(model_weights)

<All keys matched successfully>

In [6]:
import torch
import numpy as np
from sklearn.cross_decomposition import CCA

def perform_polar_cca(hidden_states_np, z1_real, z1_imag, z2_real, z2_imag):
    z1 = z1_real + 1j * z1_imag
    z2 = z2_real + 1j * z2_imag
    z_out = z1 * z2
    
    r_out = np.abs(z_out)
    phi_out = np.angle(z_out)
    
    log_r = np.log(r_out + 1e-8)
    sin_phi = np.sin(phi_out)
    cos_phi = np.cos(phi_out)
    
    Y = np.column_stack((log_r, sin_phi, cos_phi))
    X = hidden_states_np
    
    x_vars = np.var(X, axis=0)
    active_neurons = x_vars > 1e-6
    X = X[:, active_neurons]
    
    cca = CCA(n_components=3)
    cca.fit(X, Y)
    
    X_c, Y_c = cca.transform(X, Y)
    
    correlations = [np.corrcoef(X_c[:, i], Y_c[:, i])[0, 1] for i in range(3)]
    
    return correlations

def evaluate_and_run_cca(model, dataloader, device):
    model.eval()
    
    all_hidden_states = []
    all_a = []
    all_b = []
    all_c = []
    all_d = []
    
    offset = dataloader.dataset.offset
    
    with torch.no_grad():
        for batch_x, _ in dataloader:
            batch_x = batch_x.to(device)
            
            raw_a = (batch_x[:, 0] - offset).cpu().numpy()
            raw_b = (batch_x[:, 1] - offset).cpu().numpy()
            raw_c = (batch_x[:, 2] - offset).cpu().numpy()
            raw_d = (batch_x[:, 3] - offset).cpu().numpy()
            
            pos = torch.arange(batch_x.shape[1], device=device).unsqueeze(0)
            X = model.tokens_embed(batch_x) + model.pos_embed(pos)
            
            for layer in model.layers:
                X = layer(X)
                
            out = X[:, -1, :]
            
            all_hidden_states.append(out.cpu().numpy())
            all_a.extend(raw_a)
            all_b.extend(raw_b)
            all_c.extend(raw_c)
            all_d.extend(raw_d)
            
    hidden_states_np = np.concatenate(all_hidden_states, axis=0)
    z1_real = np.array(all_a)
    z1_imag = np.array(all_b)
    z2_real = np.array(all_c)
    z2_imag = np.array(all_d)
    
    correlations = perform_polar_cca(
        hidden_states_np, 
        z1_real, z1_imag, 
        z2_real, z2_imag
    )
    
    return correlations

In [7]:
import torch
from torch.utils.data import DataLoader

dataset = DatasetCN(N=N)

dataloader = DataLoader(
    dataset, 
    batch_size=256, 
    shuffle=False, 
    drop_last=False
)

device = torch.device("cpu")


print(evaluate_and_run_cca(model, dataloader=dataloader, device=device))

[np.float64(0.9506120886992498), np.float64(0.5707079641717717), np.float64(0.9415023689954143)]


In [8]:
import numpy as np
import plotly.graph_objects as go
from sklearn.decomposition import PCA

def plot_output_head_plotly_3d(model, N, head_idx=0):
    weights = model.output_heads[head_idx].weight.detach().cpu().numpy()
    
    output_offset = 2 * N * N
    min_val, max_val = -2 * N * N, 2 * N * N
    active_indices = []
    labels = []
    
    for val in range(min_val, max_val + 1):
        idx = val + output_offset
        if 0 <= idx < weights.shape[0]:
            active_indices.append(idx)
            labels.append(val)

    active_weights = weights[active_indices]

    weight_norms = np.linalg.norm(active_weights, axis=1)
    valid_mask = weight_norms > 0.4
    
    active_weights = active_weights[valid_mask]
    labels = np.array(labels)[valid_mask]


    pca = PCA(n_components=3)
    embeds_3d = pca.fit_transform(active_weights)

    # Po dopasowaniu PCA na aktywacjach
    explained_variance = pca.explained_variance_ratio_
    print(f"Komponent 1: {explained_variance[0]:.2%}")
    print(f"Komponent 2: {explained_variance[1]:.2%}")
    print(f"Komponent 3: {explained_variance[2]:.2%}")
    print(f"Suma zachowanej wariancji w 3D: {sum(explained_variance):.2%}")
    
    head_names = {0: "Re (Head 0)", 1: "Im (Head 1)"}
    title_name = head_names.get(head_idx, f"Head {head_idx}")

    fig = go.Figure()

    fig.add_trace(go.Scatter3d(
        x=embeds_3d[:, 0],
        y=embeds_3d[:, 1],
        z=embeds_3d[:, 2],
        mode='markers+text',
        text=[str(l) for l in labels],
        textposition="top right",
        textfont=dict(size=9, family="Arial Black"),
        marker=dict(
            size=6,
            color=labels,
            colorscale='RdBu',
            showscale=True,
            colorbar=dict(title=f"Wartość {title_name.split()[0]}"),
            line=dict(width=1, color='Black')
        ),
        hoverinfo='text',
        hovertext=[f"Token ID: {l + output_offset}<br>Wartość algebraiczna: {l}" for l in labels]
    ))

    fig.update_layout(
        title=dict(
            text=f"Interaktywna struktura 3D wyjść dla {title_name} (N={N})",
            x=0.5,
            xanchor='center'
        ),
        scene=dict(
            xaxis=dict(title="PCA 1", backgroundcolor="white", gridcolor="rgba(200,200,200,0.2)"),
            yaxis=dict(title="PCA 2", backgroundcolor="white", gridcolor="rgba(200,200,200,0.2)"),
            zaxis=dict(title="PCA 3", backgroundcolor="white", gridcolor="rgba(200,200,200,0.2)"),
        ),
        width=1000,
        height=900,
        showlegend=False
    )
    
    fig.show()

plot_output_head_plotly_3d(model=model, N=N)

Komponent 1: 15.50%
Komponent 2: 10.66%
Komponent 3: 9.77%
Suma zachowanej wariancji w 3D: 35.93%
